# Images and Videos with OpenCV and scikit-image

In this lab, you'll detect a face in an image with scikit-image, transform an image with OpenCV (blur, rotation, edges), then track a horse in a polo video: OpenCV reads and writes the video, frame by frame, and the correlation that tracks the horse is computed with scikit-image or with OpenCV.

1. Detecting Faces
2. Transforming an Image with OpenCV
3. Reading a Video Frame by Frame
4. Tracking the Horse by Correlation
5. Writing the Annotated Video
6. Going Further: Updating the Template
7. Going Further: An OpenCV Tracking Algorithm

Exercises 1 and 2 stand alone. Exercises 3 to 7 build on each other: each one uses what the previous ones' solutions define; the variant of exercise 4 is optional. Each exercise has a cell for your code, then additional hints and its solution, collapsed: open them if needed.

Start by running the next cell, without reading it: it downloads the video, saves its first frame and defines the `show_video` function.

In [ ]:
# @title Run me: environment setup
import os
import pathlib
import shutil
import urllib.request
import zipfile

import cv2
from IPython.display import Video, display

# Source: the VOT Challenge's “polo” sequence (https://www.votchallenge.net), 111
# frames of a polo match. Shrunk to 640 × 360, they become the video polo.mp4, at
# 10 frames/s.
POLO_URL = "https://data.votchallenge.net/sequences/acc3aeade6fb699adb7ca1c28866dd00f06d8fcd02c4cff17539a3c9d718d38976f637f4499a8763ae65f93d71fcd37494747bec7bbb8955a469b9f3fd9783aa.zip"
if not pathlib.Path("polo.mp4").exists():
  urllib.request.urlretrieve(POLO_URL, "polo.zip")
  with zipfile.ZipFile("polo.zip") as archive:
    archive.extractall("polo")
  writer = cv2.VideoWriter("polo.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 10, (640, 360))
  for path in sorted(pathlib.Path("polo").glob("*.jpg")):
    frame = cv2.resize(cv2.imread(str(path)), (640, 360), interpolation=cv2.INTER_AREA)
    writer.write(frame)
  writer.release()
  shutil.rmtree("polo")
  pathlib.Path("polo.zip").unlink()
if not pathlib.Path("polo.jpg").exists():
  # The video's first frame, for exercise 2.
  capture = cv2.VideoCapture("polo.mp4")
  ok, frame = capture.read()
  capture.release()
  cv2.imwrite("polo.jpg", frame)


def show_video(path: str) -> None:
  """Show a video in the notebook, re-encoded as WebM for the browser."""
  capture = cv2.VideoCapture(path)
  fps = capture.get(cv2.CAP_PROP_FPS)
  ok, frame = capture.read()
  height, width = frame.shape[:2]
  # Each video shown is saved in the notebook: halved, it weighs three times
  # less, and the browser enlarges it.
  size = (width // 2, height // 2)
  # OpenCV warns that the WebM format has no codec tags, then writes VP8 all
  # the same: its warning is hidden.
  stderr = os.dup(2)
  devnull = os.open(os.devnull, os.O_WRONLY)
  os.dup2(devnull, 2)
  writer = cv2.VideoWriter(
    "displayed_video.webm", cv2.VideoWriter_fourcc(*"VP80"), fps, size
  )
  os.dup2(stderr, 2)
  os.close(devnull)
  os.close(stderr)
  while ok:
    writer.write(cv2.resize(frame, size, interpolation=cv2.INTER_AREA))
    ok, frame = capture.read()
  capture.release()
  writer.release()
  display(Video("displayed_video.webm", embed=True, mimetype="video/webm", width=640))

## Imports

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches
from skimage import color, data, feature

## 1. Detecting Faces

A cascade detector (the Viola–Jones method) slides a window over the image, at several scales, and runs each position through a sequence of very simple classifiers: most windows are rejected by the first ones, and only those that pass the whole cascade are kept as faces. scikit-image provides such a detector, `feature.Cascade`, and the parameters of a model trained on frontal faces.

Detect the face in the image `data.astronaut()`, a photo of the astronaut Eileen Collins:

- create the detector from the parameter file whose path `data.lbp_frontal_face_cascade_filename()` gives;
- call its `detect_multi_scale` method on the image, with a minimum and a maximum size for the faces to look for;
- display the image with matplotlib, each detected face surrounded by a rectangle (`patches.Rectangle`).

To compare two size ranges, turn the detection and the display into a function `show_faces(image, min_size, max_size)`, which also prints the number of faces found. Then widen the range of sizes looked for, from 20 to 300 pixels: what happens?

**Useful documentation**

- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.Cascade
- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.Cascade.detect_multi_scale
- https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.lbp_frontal_face_cascade_filename
- https://matplotlib.org/stable/api/_as_gen/matplotlib.patches.Rectangle.html

In [ ]:
# Your code here

### Additional hints

- `feature.Cascade(path)` loads the model.
- `detect_multi_scale(img=image, scale_factor=1.2, step_ratio=1, min_size=(60, 60), max_size=(123, 123))`: `scale_factor` is the ratio between two successive scales of the window, `step_ratio` the step it slides by (`1`: every pixel), `min_size` and `max_size` the sizes (height, width) of the faces to look for, in pixels.
- Each detection is a dictionary: `r` and `c` are the row and column of its top-left corner, `width` and `height` its size. With the parameters above, you should find a single face, 90 × 90 pixels, at `r=69, c=177`.
- `patches.Rectangle((x, y), width, height, fill=False, edgecolor="red")` expects the abscissa `x`, the column, before the ordinate `y`, the row; `ax.add_patch(rectangle)` adds it to the figure.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
detector = feature.Cascade(...)
astronaut = data.astronaut()


def show_faces(
  image: np.ndarray, min_size: tuple[int, int], max_size: tuple[int, int]
) -> None:
  detections = detector.detect_multi_scale(
    img=image, scale_factor=1.2, step_ratio=1, min_size=..., max_size=...
  )
  print(f"{len(detections)} face(s) detected")
  fig, ax = plt.subplots()
  ax.imshow(image)
  for detection in detections:
    rectangle = patches.Rectangle(
      (..., ...),
      detection["width"],
      detection["height"],
      fill=False,
      edgecolor="red",
      linewidth=2,
    )
    ax.add_patch(rectangle)
  ax.axis("off")
  plt.show()


show_faces(astronaut, (60, 60), (123, 123))
show_faces(astronaut, ..., ...)
```

### Solution

In [ ]:
detector = feature.Cascade(data.lbp_frontal_face_cascade_filename())
astronaut = data.astronaut()


def show_faces(
  image: np.ndarray, min_size: tuple[int, int], max_size: tuple[int, int]
) -> None:
  detections = detector.detect_multi_scale(
    img=image, scale_factor=1.2, step_ratio=1, min_size=min_size, max_size=max_size
  )
  print(f"{len(detections)} face(s) detected")
  fig, ax = plt.subplots()
  ax.imshow(image)
  for detection in detections:
    rectangle = patches.Rectangle(
      (detection["c"], detection["r"]),
      detection["width"],
      detection["height"],
      fill=False,
      edgecolor="red",
      linewidth=2,
    )
    ax.add_patch(rectangle)
  ax.axis("off")
  plt.show()


show_faces(astronaut, (60, 60), (123, 123))
# A wider size range tests more windows: the computation takes longer, and the
# detector mistakes other patterns for faces.
show_faces(astronaut, (20, 20), (300, 300))

## 2. Transforming an Image with OpenCV

This exercise stands alone. The setup cell saved `polo.jpg`, the first frame of the polo video of exercises 3 to 7. With OpenCV:

- load `polo.jpg` with `cv2.imread`, and print the image's shape and type. What does `cv2.imread` return for a file that doesn't exist, `polo.jpeg` for instance?
- blur the image with `cv2.GaussianBlur`;
- rotate it by 15 degrees around its center, with `cv2.getRotationMatrix2D` and `cv2.warpAffine`;
- extract the image's edges with `cv2.Canny`, in grayscale (`cv2.cvtColor`), then those of the blurred image, and count the edge pixels of each with `np.count_nonzero`: what does the blur change?
- save the edges of the blurred image to `polo_edges.png` with `cv2.imwrite`;
- display the image, the rotated image and the two edge images with matplotlib.

**Useful documentation**

- https://numpy.org/doc/stable/reference/generated/numpy.count_nonzero.html

In [ ]:
# Your code here

### Additional hints

- `cv2.imread(path)` returns the image in BGR, an array of `uint8` of shape `(360, 640, 3)`. For a missing or unreadable file, it raises no error: it prints a warning and returns `None`, which only makes the next line fail.
- `cv2.GaussianBlur(image, (9, 9), 0)`: a 9 × 9-pixel window, of odd size, and a standard deviation of `0`, which the function derives from that size.
- `cv2.getRotationMatrix2D((width / 2, height / 2), 15, 1.0)` takes the center `(x, y)`, the angle in degrees (positive: counterclockwise) and the scale. `cv2.warpAffine(image, M, (width, height))` applies the matrix `M`; the size of the output image is written `(width, height)`, the reverse of `image.shape`.
- `cv2.Canny(gray, 100, 200)` expects a grayscale image (`cv2.COLOR_BGR2GRAY`): a pixel whose gradient exceeds 200 is an edge, a pixel between 100 and 200 is one if it touches an edge. You should find about 8,600 edge pixels without blur, and 1,800 with it.
- `cv2.imwrite(path, image)` returns `True` when the image is saved; the path's extension chooses the format.
- matplotlib expects RGB: convert the color images with `cv2.COLOR_BGR2RGB` before `imshow`, and display the edges with `cmap="gray"`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
image = cv2.imread("polo.jpg")
print(image.shape, image.dtype)
print(cv2.imread("polo.jpeg"))
height, width = image.shape[:2]

blurred = cv2.GaussianBlur(image, ..., 0)
M = cv2.getRotationMatrix2D(..., 15, 1.0)
rotated = cv2.warpAffine(image, M, ...)

edges = cv2.Canny(cv2.cvtColor(image, ...), 100, 200)
blurred_edges = cv2.Canny(...)
print(np.count_nonzero(edges), np.count_nonzero(blurred_edges))
print(cv2.imwrite("polo_edges.png", ...))

views = {
  "Image": cv2.cvtColor(image, ...),
  "Rotated by 15°": ...,
  "Edges": edges,
  "Edges after the blur": blurred_edges,
}
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, (title, view) in zip(axes.flat, views.items(), strict=True):
  ax.imshow(view, cmap="gray")
  ax.set_title(title)
  ax.axis("off")
plt.show()
```

### Solution

In [ ]:
image = cv2.imread("polo.jpg")
print(image.shape, image.dtype)
# For a missing file, imread raises no error: it returns None.
print(cv2.imread("polo.jpeg"))
height, width = image.shape[:2]

blurred = cv2.GaussianBlur(image, (9, 9), 0)
M = cv2.getRotationMatrix2D((width / 2, height / 2), 15, 1.0)
rotated = cv2.warpAffine(image, M, (width, height))

edges = cv2.Canny(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), 100, 200)
blurred_edges = cv2.Canny(cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY), 100, 200)
# The blur erases the foliage's details: what is left are the strong edges,
# those of the horses, of the field's boundary and of the car.
print(np.count_nonzero(edges), np.count_nonzero(blurred_edges))
print(cv2.imwrite("polo_edges.png", blurred_edges))

views = {
  "Image": cv2.cvtColor(image, cv2.COLOR_BGR2RGB),
  "Rotated by 15°": cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB),
  "Edges": edges,
  "Edges after the blur": blurred_edges,
}
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, (title, view) in zip(axes.flat, views.items(), strict=True):
  ax.imshow(view, cmap="gray")
  ax.set_title(title)
  ax.axis("off")
plt.show()

## A Polo Video

The setup cell created `polo.mp4`: the 111 frames of a polo match from the VOT Challenge's `polo` sequence (a competition of object tracking algorithms), shrunk to 640 × 360 pixels, at 10 frames per second. It also defined `show_video(path)`, which shows a video in the notebook. Watch the video: you'll track the horse on the left.

In [ ]:
show_video("polo.mp4")

## 3. Reading a Video Frame by Frame

Write the function `read_frames(path)`, which reads a video with `cv2.VideoCapture`, as in the course, and returns the list of its frames, each one converted from BGR, OpenCV's channel order, to RGB, the one of matplotlib and scikit-image, as well as its frame rate, in frames per second, which the `get` method of `cv2.VideoCapture` gives: the annotated video of exercise 5 will reuse it.

Read `polo.mp4` into `frames` and `fps`, print the number of frames, the shape of one and the frame rate, then display the first one with matplotlib, with the rectangle `HORSE_BOX`: it is the box of the horse to track. A box is a tuple `(x, y, width, height)`, `(x, y)` being its top-left corner.

**Useful documentation**

- https://docs.opencv.org/4.x/d8/dfe/classcv_1_1VideoCapture.html

In [ ]:
HORSE_BOX = (160, 160, 80, 73)  # (x, y, width, height)


def read_frames(path: str) -> tuple[list[np.ndarray], float]:
  # Your code here
  ...

### Additional hints

- `capture.read()` returns a pair `(ok, image)`: `ok` is `False` once the video is over, and the image is then `None`.
- `cv2.cvtColor(image, cv2.COLOR_BGR2RGB)` converts an image from BGR to RGB.
- `capture.get(cv2.CAP_PROP_FPS)` returns the video's frame rate, a float.
- You should get 111 frames of shape `(360, 640, 3)`, at 10 frames per second (`10.0`).

A skeleton to copy into the exercise cell, replacing each `...`:

```python
HORSE_BOX = (160, 160, 80, 73)


def read_frames(path: str) -> tuple[list[np.ndarray], float]:
  capture = cv2.VideoCapture(path)
  fps = ...
  frames = []
  ok, image = capture.read()
  while ok:
    frames.append(...)
    ok, image = capture.read()
  capture.release()
  return frames, fps


frames, fps = read_frames("polo.mp4")
print(len(frames), frames[0].shape, fps)

x, y, width, height = HORSE_BOX
fig, ax = plt.subplots()
ax.imshow(...)
ax.add_patch(patches.Rectangle(...))
plt.show()
```

### Solution

In [ ]:
HORSE_BOX = (160, 160, 80, 73)  # (x, y, width, height)


def read_frames(path: str) -> tuple[list[np.ndarray], float]:
  capture = cv2.VideoCapture(path)
  fps = capture.get(cv2.CAP_PROP_FPS)
  frames = []
  ok, image = capture.read()
  while ok:
    # OpenCV reads frames as BGR, matplotlib and scikit-image want RGB.
    frames.append(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ok, image = capture.read()
  capture.release()
  return frames, fps


frames, fps = read_frames("polo.mp4")
print(len(frames), frames[0].shape, fps)

x, y, width, height = HORSE_BOX
fig, ax = plt.subplots()
ax.imshow(frames[0])
ax.add_patch(
  patches.Rectangle((x, y), width, height, fill=False, edgecolor="red", linewidth=2)
)
plt.show()

## 4. Tracking the Horse by Correlation

The simplest tracking looks for a template, the patch of the horse in the first frame, in each of the next frames. `feature.match_template(region, template)` computes, for each position of the template in a region of the frame, the normalized correlation between the template and the frame, between -1 and 1: its maximum shows where the frame looks most like the template.

Write the function `track(frames, box, margin=30)`, which returns the horse's box in each frame:

- the template is the content of `box` in the first frame, in grayscale;
- in each next frame, look for the template in a region that extends the previous box by `margin` pixels on each side, and put the new box at the position of the correlation's maximum.

Track the horse from `HORSE_BOX` and print one box in ten. Then try a 10-pixel margin, then a margin covering the whole frame (1000 pixels): what happens? Time the computation with `%time`.

**Useful documentation**

- https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.match_template
- https://numpy.org/doc/stable/reference/generated/numpy.unravel_index.html

In [ ]:
def track(
  frames: list[np.ndarray], box: tuple[int, int, int, int], margin: int = 30
) -> list[tuple[int, int, int, int]]:
  # Your code here
  ...

### Additional hints

- `color.rgb2gray(image)` converts an RGB image to grayscale, floats between 0 and 1. The template is `image[y : y + height, x : x + width]`: rows first, then columns.
- The region starts at `(x0, y0) = (max(x - margin, 0), max(y - margin, 0))`, so as not to leave the frame, and ends `margin` pixels after the box (a NumPy slice that goes too far stops at the frame's edge).
- `feature.match_template(region, template)` returns an array of correlations, one per position of the template's top-left corner in the region. `np.unravel_index(np.argmax(scores), scores.shape)` gives the row and column `(dy, dx)` of its maximum: the new box starts at `(x0 + dx, y0 + dy)`.
- With a 30-pixel margin, the last box is close to `(189, 191, 80, 73)`.
- `%time` before a line prints the time it takes.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def track(
  frames: list[np.ndarray], box: tuple[int, int, int, int], margin: int = 30
) -> list[tuple[int, int, int, int]]:
  x, y, width, height = box
  template = color.rgb2gray(...)
  boxes = [box]
  for image in frames[1:]:
    x0, y0 = max(x - margin, 0), max(y - margin, 0)
    region = color.rgb2gray(image[y0 : y + height + margin, x0 : x + width + margin])
    scores = feature.match_template(region, template)
    dy, dx = ...
    x, y = int(x0 + dx), int(y0 + dy)
    boxes.append((x, y, width, height))
  return boxes


%time boxes = track(frames, HORSE_BOX)
print(boxes[::10])
print(track(frames, HORSE_BOX, margin=10)[::10])
%time print(track(frames, HORSE_BOX, margin=1000)[::10])
```

### Solution

In [ ]:
def track(
  frames: list[np.ndarray], box: tuple[int, int, int, int], margin: int = 30
) -> list[tuple[int, int, int, int]]:
  x, y, width, height = box
  template = color.rgb2gray(frames[0][y : y + height, x : x + width])
  boxes = [box]
  for image in frames[1:]:
    # The search region: the previous box, widened by the margin.
    x0, y0 = max(x - margin, 0), max(y - margin, 0)
    region = color.rgb2gray(image[y0 : y + height + margin, x0 : x + width + margin])
    scores = feature.match_template(region, template)
    dy, dx = np.unravel_index(np.argmax(scores), scores.shape)
    x, y = int(x0 + dx), int(y0 + dy)
    boxes.append((x, y, width, height))
  return boxes


%time boxes = track(frames, HORSE_BOX)
print(boxes[::10])
# A 10-pixel margin loses the horse, which moves faster than that.
print(track(frames, HORSE_BOX, margin=10)[::10])
# A margin covering the whole frame is slower, and the box ends up jumping to
# the other horse, which looks like the template too.
%time print(track(frames, HORSE_BOX, margin=1000)[::10])

### Variant: OpenCV's Correlation

This variant is optional: what follows uses the boxes of `track`. OpenCV computes the same normalized correlation with `cv2.matchTemplate(region, template, cv2.TM_CCOEFF_NORMED)`, and `cv2.minMaxLoc` gives the position of its maximum.

Write the function `track_opencv(frames, box, margin=30)`, like `track`, but with these two functions, and grayscale images obtained with `cv2.cvtColor`. Compare its boxes with those of `track`, and its computation time, with a 30-pixel margin, then a 1000-pixel one.

In [ ]:
def track_opencv(
  frames: list[np.ndarray], box: tuple[int, int, int, int], margin: int = 30
) -> list[tuple[int, int, int, int]]:
  # Your code here
  ...

#### Additional hints

- `cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)` converts a frame of the list, which is in RGB, to grayscale; the result is `uint8`, which `cv2.matchTemplate` accepts as well as `float32`.
- `cv2.minMaxLoc(scores)` returns `(minimum, maximum, min_position, max_position)`, each position as `(x, y)`: the column first, unlike `np.unravel_index`.
- With a 30-pixel margin, the boxes are those of `track` except in about fifteen frames, where they stray a little, and the last one is the same, `(189, 191, 80, 73)`: the two libraries don't weight the channels the same way when converting to grayscale. The computation is much faster with OpenCV.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def track_opencv(
  frames: list[np.ndarray], box: tuple[int, int, int, int], margin: int = 30
) -> list[tuple[int, int, int, int]]:
  x, y, width, height = box
  template = cv2.cvtColor(frames[0][y : y + height, x : x + width], ...)
  boxes = [box]
  for image in frames[1:]:
    x0, y0 = max(x - margin, 0), max(y - margin, 0)
    region = cv2.cvtColor(image[y0 : y + height + margin, x0 : x + width + margin], ...)
    scores = cv2.matchTemplate(region, template, cv2.TM_CCOEFF_NORMED)
    _, _, _, (dx, dy) = ...
    x, y = x0 + dx, y0 + dy
    boxes.append((x, y, width, height))
  return boxes


%time boxes_opencv = track_opencv(frames, HORSE_BOX)
print(boxes_opencv[::10])
print(sum(a != b for a, b in zip(boxes, boxes_opencv, strict=True)), "different boxes")
%time print(track_opencv(frames, HORSE_BOX, margin=1000)[::10])
```

#### Solution

In [ ]:
def track_opencv(
  frames: list[np.ndarray], box: tuple[int, int, int, int], margin: int = 30
) -> list[tuple[int, int, int, int]]:
  x, y, width, height = box
  template = cv2.cvtColor(frames[0][y : y + height, x : x + width], cv2.COLOR_RGB2GRAY)
  boxes = [box]
  for image in frames[1:]:
    x0, y0 = max(x - margin, 0), max(y - margin, 0)
    region = cv2.cvtColor(
      image[y0 : y + height + margin, x0 : x + width + margin], cv2.COLOR_RGB2GRAY
    )
    scores = cv2.matchTemplate(region, template, cv2.TM_CCOEFF_NORMED)
    # minMaxLoc gives the positions as (x, y), the column first.
    _, _, _, (dx, dy) = cv2.minMaxLoc(scores)
    x, y = x0 + dx, y0 + dy
    boxes.append((x, y, width, height))
  return boxes


%time boxes_opencv = track_opencv(frames, HORSE_BOX)
print(boxes_opencv[::10])
# A few boxes differ from those of track: OpenCV's grayscale conversion doesn't
# weight the channels like scikit-image's.
print(sum(a != b for a, b in zip(boxes, boxes_opencv, strict=True)), "different boxes")
# The same computation, much faster.
%time print(track_opencv(frames, HORSE_BOX, margin=1000)[::10])

## 5. Writing the Annotated Video

`cv2.VideoWriter` writes a video frame by frame, as `cv2.VideoCapture` reads one. Write the function `write_video(frames, boxes, path, fps)`, which writes to `path` the video of the frames, at `fps` frames per second, each one with its box drawn by `cv2.rectangle`. Write `polo_tracked.mp4` with the boxes of the previous exercise, at the frame rate of `polo.mp4` read in exercise 3, then show it with `show_video`.

Two pitfalls: OpenCV writes BGR frames, and `cv2.rectangle` draws in the image it is given. Draw in a copy, to keep the list's frames intact.

**Useful documentation**

- https://docs.opencv.org/4.x/dd/d9e/classcv_1_1VideoWriter.html
- https://docs.opencv.org/4.x/d6/d6e/group__imgproc__draw.html#ga07d2f74cadcf8e305e810ce8eed13bc9

In [ ]:
def write_video(
  frames: list[np.ndarray],
  boxes: list[tuple[int, int, int, int]],
  path: str,
  fps: float,
) -> None:
  # Your code here
  ...

### Additional hints

- `cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))` creates the video: its size is `(width, height)`, the reverse of `image.shape`'s order. The FourCC (four character code) names the codec in four characters, here `mp4v`, MPEG-4; the other codes: https://fourcc.org/codecs.php.
- `writer.write(image)` adds a frame, in BGR, and `writer.release()` finishes the file.
- `cv2.rectangle(image, (x, y), (x + width, y + height), color=(255, 0, 0), thickness=2)` takes two opposite corners; the color follows the image's channel order: `(255, 0, 0)` is red in an RGB image.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def write_video(
  frames: list[np.ndarray],
  boxes: list[tuple[int, int, int, int]],
  path: str,
  fps: float,
) -> None:
  frame_height, frame_width = frames[0].shape[:2]
  writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (..., ...))
  for image, (x, y, width, height) in zip(frames, boxes, strict=True):
    annotated = image.copy()
    cv2.rectangle(annotated, ..., ..., color=(255, 0, 0), thickness=2)
    writer.write(cv2.cvtColor(annotated, ...))
  writer.release()


write_video(frames, boxes, "polo_tracked.mp4", fps)
show_video("polo_tracked.mp4")
```

### Solution

In [ ]:
def write_video(
  frames: list[np.ndarray],
  boxes: list[tuple[int, int, int, int]],
  path: str,
  fps: float,
) -> None:
  frame_height, frame_width = frames[0].shape[:2]
  writer = cv2.VideoWriter(
    path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame_width, frame_height)
  )
  for image, (x, y, width, height) in zip(frames, boxes, strict=True):
    # A copy, so as not to draw the box in the list's frames.
    annotated = image.copy()
    cv2.rectangle(
      annotated, (x, y), (x + width, y + height), color=(255, 0, 0), thickness=2
    )
    writer.write(cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))
  writer.release()


write_video(frames, boxes, "polo_tracked.mp4", fps)
show_video("polo_tracked.mp4")

## 6. Going Further: Updating the Template

The horse turns and changes its look over the video, while the template stays the one from the first frame. Write `track_updating(frames, box, margin=30)`, like `track`, but replacing the template, at each frame, with the content of the box found. Write and show the video of this tracking, `polo_updated.mp4`: does the box follow the horse better? Why?

In [ ]:
def track_updating(
  frames: list[np.ndarray], box: tuple[int, int, int, int], margin: int = 30
) -> list[tuple[int, int, int, int]]:
  # Your code here
  ...

### Additional hints

Only one line changes from `track`: once the new box is found, the template becomes its content in the current frame, in grayscale. Compare the last boxes of the two trackings, close to `(189, 191, 80, 73)` without updates and to `(165, 175, 80, 73)` with them, then look at where the box is at the end of the video.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def track_updating(
  frames: list[np.ndarray], box: tuple[int, int, int, int], margin: int = 30
) -> list[tuple[int, int, int, int]]:
  x, y, width, height = box
  template = color.rgb2gray(frames[0][y : y + height, x : x + width])
  boxes = [box]
  for image in frames[1:]:
    x0, y0 = max(x - margin, 0), max(y - margin, 0)
    region = color.rgb2gray(image[y0 : y + height + margin, x0 : x + width + margin])
    scores = feature.match_template(region, template)
    dy, dx = np.unravel_index(np.argmax(scores), scores.shape)
    x, y = int(x0 + dx), int(y0 + dy)
    template = ...
    boxes.append((x, y, width, height))
  return boxes


updated_boxes = track_updating(frames, HORSE_BOX)
print(updated_boxes[::10])
write_video(frames, updated_boxes, "polo_updated.mp4", fps)
show_video("polo_updated.mp4")
```

### Solution

In [ ]:
def track_updating(
  frames: list[np.ndarray], box: tuple[int, int, int, int], margin: int = 30
) -> list[tuple[int, int, int, int]]:
  x, y, width, height = box
  template = color.rgb2gray(frames[0][y : y + height, x : x + width])
  boxes = [box]
  for image in frames[1:]:
    x0, y0 = max(x - margin, 0), max(y - margin, 0)
    region = color.rgb2gray(image[y0 : y + height + margin, x0 : x + width + margin])
    scores = feature.match_template(region, template)
    dy, dx = np.unravel_index(np.argmax(scores), scores.shape)
    x, y = int(x0 + dx), int(y0 + dy)
    # The new template: the content of the box found.
    template = color.rgb2gray(image[y : y + height, x : x + width])
    boxes.append((x, y, width, height))
  return boxes


updated_boxes = track_updating(frames, HORSE_BOX)
print(updated_boxes[::10])
# The box drifts: each template carries over the previous one's small position
# error and a bit of background, the errors add up, and the box slides off the
# horse little by little.
write_video(frames, updated_boxes, "polo_updated.mp4", fps)
show_video("polo_updated.mp4")

## 7. Going Further: An OpenCV Tracking Algorithm

OpenCV provides more elaborate tracking algorithms, which learn the object's look over the video. `cv2.TrackerMIL` (Multiple Instance Learning) trains a classifier to tell the object from its surroundings, and updates it at every frame. Write `track_mil(frames, box)`, which returns the boxes this algorithm finds, then write and show its video, `polo_mil.mp4`. Compare its tracking and its computation time with the correlation's.

**Useful documentation**

- https://docs.opencv.org/4.x/d0/d26/classcv_1_1TrackerMIL.html
- https://docs.opencv.org/4.x/d0/d0a/classcv_1_1Tracker.html

In [ ]:
def track_mil(
  frames: list[np.ndarray], box: tuple[int, int, int, int]
) -> list[tuple[int, int, int, int]]:
  # Your code here
  ...

### Additional hints

- `tracker = cv2.TrackerMIL_create()` creates the algorithm, `tracker.init(image, box)` initializes it on the first frame and its box, then `ok, found = tracker.update(image)` gives the box in each next frame. `ok` is `False` when the algorithm has lost the object: keep the previous box then.
- OpenCV expects BGR frames, and `found` may hold floats: convert them with `int`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def track_mil(
  frames: list[np.ndarray], box: tuple[int, int, int, int]
) -> list[tuple[int, int, int, int]]:
  tracker = cv2.TrackerMIL_create()
  tracker.init(cv2.cvtColor(frames[0], cv2.COLOR_RGB2BGR), box)
  boxes = [box]
  for image in frames[1:]:
    ok, found = tracker.update(...)
    if ok:
      box = ...
    boxes.append(box)
  return boxes


%time mil_boxes = track_mil(frames, HORSE_BOX)
write_video(frames, mil_boxes, "polo_mil.mp4", fps)
show_video("polo_mil.mp4")
```

### Solution

In [ ]:
def track_mil(
  frames: list[np.ndarray], box: tuple[int, int, int, int]
) -> list[tuple[int, int, int, int]]:
  tracker = cv2.TrackerMIL_create()
  # The list's frames are RGB, OpenCV expects BGR.
  tracker.init(cv2.cvtColor(frames[0], cv2.COLOR_RGB2BGR), box)
  boxes = [box]
  for image in frames[1:]:
    ok, found = tracker.update(cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    # When the tracker loses the object, the box stays where it was.
    if ok:
      x, y, width, height = (int(v) for v in found)
      box = (x, y, width, height)
    boxes.append(box)
  return boxes


%time mil_boxes = track_mil(frames, HORSE_BOX)
print(mil_boxes[::10])
# MIL follows the horse too, but much more slowly than the
# correlation: it trains a classifier on every frame.
write_video(frames, mil_boxes, "polo_mil.mp4", fps)
show_video("polo_mil.mp4")